In [1]:
import os
from dotenv import load_dotenv

load_dotenv()  # reads your .env file

print("✅ Tracing is active. Ready to send data to LangSmith.")

✅ Tracing is active. Ready to send data to LangSmith.


In [1]:
import os
from dotenv import load_dotenv
load_dotenv()

from langchain_community.document_loaders import DirectoryLoader, CSVLoader, TextLoader, PyPDFLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS
from langchain.schema import Document
from langchain_groq import ChatGroq
from langchain_core.prompts import ChatPromptTemplate
from langchain.chains.combine_documents import create_stuff_documents_chain
from langchain.chains import create_retrieval_chain

# ✅ Local paths — replace with the actual full path to your Dalil folder
BASE_PATH       = r"C:\Users\yasse\Dalil\data" 
SOURCE_DATA_PATH = os.path.join(BASE_PATH, "Source_data")
VECTOR_DB_PATH   = os.path.join(BASE_PATH, "vector_db_index")
LOG_FILE_PATH    = os.path.join(BASE_PATH, "admin_knowledge_base.txt")

# LLM — reads GROQ_API_KEY automatically from .env
llm = ChatGroq(model="llama-3.1-8b-instant", temperature=0.1)

# Embedding model
embedding_model = HuggingFaceEmbeddings(model_name="BAAI/bge-large-en-v1.5")

print("✅ Dalil Selective Engine Ready.")

C:\Users\yasse\AppData\Local\Temp\ipykernel_30792\3365955138.py:25: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-huggingface package and should be used instead. To use it run `pip install -U :class:`~langchain-huggingface` and import as `from :class:`~langchain_huggingface import HuggingFaceEmbeddings``.
  embedding_model = HuggingFaceEmbeddings(model_name="BAAI/bge-large-en-v1.5")
c:\Users\yasse\Dalil\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
c:\Users\yasse\Dalil\venv\Lib\site-packages\huggingface_hub\file_download.py:138: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not suppor

✅ Dalil Selective Engine Ready.


In [2]:
def get_or_create_vectorstore(embedding_model):
    # 1. Check if the vector database already exists for fast loading
    if os.path.exists(os.path.join(VECTOR_DB_PATH, "index.faiss")):
        print("✅ Vector database found. Loading directly from Drive...")
        return FAISS.load_local(VECTOR_DB_PATH, embedding_model, allow_dangerous_deserialization=True)

    # 2. If not found, start processing original source files (PDF, CSV, TXT)
    print("🆕 Vector database not found. Processing source files...")

    all_docs = []

    # Check if the source directory exists before loading
    if os.path.exists(SOURCE_DATA_PATH):
        # Load PDF documents
        pdf_loader = DirectoryLoader(SOURCE_DATA_PATH, glob="*.pdf", loader_cls=PyPDFLoader)
        all_docs.extend(pdf_loader.load())

        # Load CSV documents with UTF-8 encoding
        csv_loader = DirectoryLoader(SOURCE_DATA_PATH, glob="*.csv", loader_cls=CSVLoader, loader_kwargs={"encoding": "utf-8"})
        all_docs.extend(csv_loader.load())

        # Load TXT documents (including admin logs/overrides)
        txt_loader = DirectoryLoader(SOURCE_DATA_PATH, glob="*.txt", loader_cls=TextLoader, loader_kwargs={"encoding": "utf-8"})
        all_docs.extend(txt_loader.load())

    # 3. Handle cases where no files are found to avoid errors
    if not all_docs:
        print("⚠️ No files found! Creating an initial virtual database.")
        return FAISS.from_documents([Document(page_content="Initial System")], embedding_model)

    # 4. Split documents into smaller chunks for better retrieval accuracy
    text_splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50)
    chunks = text_splitter.split_documents(all_docs)

    # 5. Create the FAISS vector store and save it to Drive for future use
    vectorstore = FAISS.from_documents(chunks, embedding_model)

    os.makedirs(VECTOR_DB_PATH, exist_ok=True)
    vectorstore.save_local(VECTOR_DB_PATH)

    print(f"✅ New database created: Processed {len(all_docs)} files and generated {len(chunks)} chunks.")
    return vectorstore

# Execute the loading/creation process
vectorstore = get_or_create_vectorstore(embedding_model)

🆕 Vector database not found. Processing source files...
✅ New database created: Processed 7801 files and generated 7801 chunks.


In [3]:
retriever = vectorstore.as_retriever(search_kwargs={"k": 5})
print("✅ Retriever is ready and linked to the active Vectorstore.")

✅ Retriever is ready and linked to the active Vectorstore.


In [4]:
# 1. Selective Deletion Function (Admin Overrides Management)
def internal_selective_delete(name, category=""):
    """
    Removes previous manual updates/inserts from the log file to prevent redundancy.
    Note: This does NOT modify the original CSV or PDF source files.
    """
    global vectorstore
    if os.path.exists(LOG_FILE_PATH):
        with open(LOG_FILE_PATH, "r", encoding="utf-8") as f:
            lines = f.readlines()

        # Filter out lines that contain BOTH the name and the category (the old record)
        new_lines = [
            l for l in lines
            if not (name.lower().strip() in l.lower() and category.lower().strip() in l.lower())
        ]

        # Overwrite the log file with the cleaned-up data
        with open(LOG_FILE_PATH, "w", encoding="utf-8") as f:
            f.writelines(new_lines)
        return True
    return False

# 2. Main Dalil Engine (Supports Admin 'Insert/Delete' and User 'Search')
def dalil_main_engine(user_input, is_admin=False):
    """
    Orchestrates the system behavior based on user role and input command.
    """
    global vectorstore

    if is_admin:
        # --- DELETE COMMAND ---
        if user_input.lower().startswith("delete:"):
            target = user_input.replace("delete:", "").strip()
            internal_selective_delete(target, "")
            return f"🗑️ Records for '{target}' removed from overrides."

        # --- INSERT/UPDATE COMMAND ---
        elif user_input.lower().startswith("insert:"):
            try:
                # Expected Format: "insert: Name, Category, Value"
                content = user_input.replace("insert:", "").strip()
                parts = [p.strip() for p in content.split(",")]

                if len(parts) >= 3:
                    name, category, value = parts[0], parts[1], parts[2]

                    # Prefixing with 'UPDATED INFO' gives this record priority during AI reasoning
                    new_record = f"UPDATED INFO: {name}'s {category} is {value}"

                    # Clean up existing manual overrides for this specific person/category
                    internal_selective_delete(name, category)

                    # Add the new record to the active Vector Store and save it to Drive
                    vectorstore.add_documents([Document(page_content=new_record)])
                    vectorstore.save_local(VECTOR_DB_PATH)

                    # Persist the change in the TXT log file for future session reloads
                    with open(LOG_FILE_PATH, "a", encoding="utf-8") as f:
                        f.write(new_record + "\n")

                    return f"✅ Successfully Inserted: {name}'s {category} record."
                else:
                    return "⚠️ Format Error! Use: insert: Name, Category, Value"
            except Exception as e:
                return f"❌ System Error: {str(e)}"

    # --- USER SEARCH (Standard Retrieval) ---
    # Retrieve the top 7 most relevant chunks to provide deep context
    docs = vectorstore.similarity_search(user_input, k=5)
    context = "\n".join([d.page_content for d in docs])

    # Strict System Instructions to prioritize manual inserts over old source files
    system_instruction = (
        "You are an expert assistant. Use the provided context to answer. "
        "IMPORTANT: If you find conflicting information, prioritize records labeled 'UPDATED INFO'."
    )

    # Construct the final prompt and invoke the LLM
    final_prompt = f"{system_instruction}\n\nContext:\n{context}\n\nQuestion: {user_input}"
    return llm.invoke(final_prompt).content

In [5]:
# ==============================================================================
# PROMPT CONFIGURATION (LLM Persona & Instructions)
# This cell defines the rules and format the AI must follow when answering.
# ==============================================================================

from langchain_core.prompts import ChatPromptTemplate

# 1. Define the System Prompt (The rules)
system_prompt_text = """You are an expert university teaching assistant.

Use the retrieved context ONLY as reference.
DO NOT copy the text directly.
DO NOT list chunks.

Your task:
1. Understand the user's question.
2. Extract relevant ideas from the context.
3. Reason step by step.
4. Generate a clear, original answer in your own words.

If the context is insufficient, say so.

ALWAYS follow this format:
**Answer:**
<direct answer>

**Details:**
- <bullet point 1>
- <bullet point 2>

Context:
{context}

Question:
{input}

Answer:
"""

# 2. Initialize the ChatPromptTemplate
# This combines the system instructions with the user's dynamic input
qa_prompt = ChatPromptTemplate.from_messages([
    ("system", system_prompt_text),
    ("human", "{input}")
])

print("✅ System Prompt and QA Template are ready.")

✅ System Prompt and QA Template are ready.


In [6]:
#  RAG Document Chain + Retrieval Chain
document_chain = create_stuff_documents_chain(
    llm=llm,
    prompt=qa_prompt
)

rag_chain = create_retrieval_chain(
    retriever,
    document_chain
)

print("✅ RAG chain is ready.")

✅ RAG chain is ready.


In [7]:
# CONVERSATIONAL CHAT FUNCTION WITH MEMORY
chat_history = []

def chat(question):
    global chat_history

    result = rag_chain.invoke({
        "input": question,
        # "question":question,
        "chat_history": chat_history
    })

    answer = result["answer"]

    chat_history.append({"user": question})
    chat_history.append({"assistant": answer})

    return answer


In [8]:
print(chat("Who has the highest GPA?"))

**Answer:**
Edwin Sanders.

**Details:**
- Edwin Sanders has a GPA of 4.00, which is the highest among all students in the dataset.
- His GPA ranks him at the top of his batch, indicating exceptional academic performance.


In [9]:
print(chat("Search in all data the ten top students by GPA"))

**Answer:**
The top ten students by GPA are Edwin Sanders (4.00), followed by other students whose GPA information is not provided in the given context.

**Details:**
- The provided data only contains information about Edwin Sanders, who has the highest GPA of 4.00.
- To find the top ten students by GPA, we would need more data on other students' GPAs.
- The given data does not provide a comprehensive list of students, making it impossible to determine the exact top ten students by GPA.


In [10]:
print(chat( "What the marjon Bobby Ramirez "))

**Answer:**
Bobby Ramirez is a Science major.

**Details:**
- The context lists Bobby Ramirez's student information, which includes their major as Science.
- This information is used to determine Bobby Ramirez's major.


In [11]:
print(chat("Search in all data the five top students by GPA"))

**Answer:**
The top five students by GPA are Edwin Sanders, followed by S00275, S00375, S00495, and S00265.

**Details:**
- Edwin Sanders has the highest GPA of 4.00, which ranks him at the top of his batch.
- S00275 has a GPA of 89.7, which is the second-highest GPA in the dataset.
- S00375 has a GPA of 63.1, which ranks him third among the top five students.
- S00495 has a GPA of 5.9, which is the fourth-highest GPA in the dataset.
- S00265 has a GPA of 8.0, which ranks him fifth among the top five students.


In [12]:
print(chat( "If I missed the application deadline ,what can I do?"))

**Answer:**
You may still be eligible to apply, but you'll need to contact the admissions office to discuss possible options.

**Details:**
- The Late Application Handling policy outlines eligibility criteria and required documents, which suggests that there may be flexibility in the application process.
- Exceptions may apply for transfer or international students, indicating that individual circumstances can be taken into account.
- The multiple versions of the Late Application Handling policy with varying effective dates and last updated dates suggest that the university is willing to adapt and update its policies to accommodate changing circumstances.


In [13]:
chat("ما هي وظيفة مشروع دليل؟")

'**Answer:**\nمشروع دليل هو وظيفة تهدف إلى توفير المعلومات والتعليمات اللازمة لتحقيق النجاح في المقرر.\n\n**Details:**\n- يعد مشروع دليل مصدرًا هامًا للمعلمين والطلاب في فهم المقرر وتنفيذه.\n- يحتوي على وصف للمقرر، ووصف للمهارات والنتائج التي يجب تحقيقها، ووصف للمهام والأنشطة التي يجب إنجازها.\n- يساعد الطلاب في فهم الأهداف والمتطلبات المقررة، ويساعد المعلمين في توجيه الطلاب نحو تحقيق النجاح في المقرر.'

In [14]:
print(dalil_main_engine("update: Olivia McKinney, GPA, 3.8", is_admin=True))

Based on the provided context, I will update Olivia McKinney's information.

Before update:
- student_id: S00010
- name: Olivia Mckinney
- gender: Male
- age: 28
- major: Science
- year: 1
- GPA: 2.07
- status: Active

After update:
- student_id: S00010
- name: Olivia Mckinney
- gender: Male
- age: 28
- major: Science
- year: 1
- GPA: 3.8
- status: Active

The update has been successfully applied.


In [15]:
print("Address Check:", dalil_main_engine("what is the majon Olivia McKinney?", is_admin=False))

Address Check: There are multiple students with the name "Olivia". To provide an accurate answer, I'll need to identify the correct student. 

Based on the provided context, I found two students with the name "Olivia":

1. Olivia Mckinney (student_id: S00010) - major: Science
2. Olivia Juarez (student_id: S01209) - major: Computer Science
3. Olivia Brown (student_id: S00739) - major: Law

Since the question asks for the major of "Olivia McKinney", I will assume it refers to Olivia Mckinney (student_id: S00010). 

The major of Olivia McKinney is Science.


In [16]:
print(dalil_main_engine("insert: Habiba Yasser, Student Profile, student_id=s01501 | age=21 | major=computer science | year=4 | GPA=3.0 | status=Active", is_admin=True))

✅ Successfully Inserted: Habiba Yasser's Student Profile record.


In [17]:
print(dalil_main_engine("What is the GPA of student habiba yasser?", is_admin=False))

According to the provided context, specifically the 'UPDATED INFO' section, Habiba Yasser's Student Profile is:

student_id = s01501
GPA = 3.0

So, the GPA of student Habiba Yasser is 3.0.
